In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 27.31 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Dataset Verification, Head Inspection

import os
import glob
import pandas as pd

# 1. Locate dataset CSV
possible_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(possible_csv) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# 2. Map image index to full filesystem path across all image folders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# 3. 14 standard thorax pathologies in NIH-14
CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

# 4. Official train/test split using test_list.txt
test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)

print(f"\n✓ Verified Images Available: {len(df)}")
print(f"✓ Train Pool: {len(train_df)} scans | Test Pool: {len(test_df)} scans")

# Display the head of the processed dataset
df[['Image Index', 'Finding Labels'] + CLASSES[:5]].head(5)

Found Metadata CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv

✓ Verified Images Available: 112120
✓ Train Pool: 86524 scans | Test Pool: 25596 scans


,Image Index,Finding Labels,Atelectasis,Cardiomegaly,Effusion,Infiltration,Mass
0,00000001_000.png,Cardiomegaly,0,1,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,0,1,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,0,1,1,0,0
3,00000002_000.png,No Finding,0,0,0,0,0
4,00000003_000.png,Hernia,0,0,0,0,0


In [3]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [4]:
import os
import sys
import glob
import importlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()
from chexfound.models import vision_transformer as vits

# 2. Check Cache or Automatically Extract
cache_file = '/kaggle/working/shenzhen_cls_features.pt'

if os.path.exists(cache_file):
    print(f"Loading existing cache from {cache_file}...")
    cached_data = torch.load(cache_file, map_location='cpu')
    X_all = cached_data['X_all']
    y_all = cached_data['y_all']
    print(f"✓ Loaded cached feature matrix: {X_all.shape}")
else:
    print(f"Cache not found in this environment. Extracting features from scratch...")
    
    # Load Frozen CheXFound ViT-Large Backbone
    teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
    assert len(teacher_files) > 0, "teacher_checkpoint.pth missing from /kaggle/input/"
    
    model = vits.vit_large(patch_size=16, img_size=512)
    teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
    state = teacher_ckpt.get('teacher', teacher_ckpt)
    clean_state = {(k[9:] if k.startswith('backbone.') else k): v for k, v in state.items()}
    model.load_state_dict(clean_state, strict=False)
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    print("✓ Backbone loaded and frozen.")

    # Locate Shenzhen scans
    search_paths = [
        '/kaggle/input/**/CXR_png/*.png',
        '/kaggle/input/**/images/*.png',
        '/kaggle/input/**/*.png'
    ]
    tb_images = []
    for sp in search_paths:
        found = glob.glob(sp, recursive=True)
        tb_images = [p for p in found if 'CHN' in os.path.basename(p) or 'CHX' in os.path.basename(p)]
        if len(tb_images) > 0:
            break

    assert len(tb_images) > 0, "Shenzhen images not found. Make sure dataset is attached."

    records = []
    for p in tb_images:
        label = 1 if os.path.splitext(os.path.basename(p))[0].endswith('_1') else 0
        records.append({'path': p, 'label': label})

    df_all = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
    y_all = torch.tensor(df_all['label'].values)

    eval_transform = T.Compose([
        T.Resize((512, 512)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    class ShenzhenDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
            return self.transform(img), self.df.iloc[idx]['label']

    all_loader = DataLoader(ShenzhenDataset(df_all, eval_transform), batch_size=16, shuffle=False, num_workers=2)

    all_cls = []
    with torch.no_grad():
        for imgs, _ in tqdm(all_loader, desc="Precomputing CLS Features"):
            imgs = imgs.to(device)
            layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
            cls_concat = torch.cat([layer[1] for layer in layers], dim=-1)
            all_cls.append(cls_concat.cpu())

    X_all = torch.cat(all_cls, dim=0)
    torch.save({'X_all': X_all, 'y_all': y_all}, cache_file)
    print(f"✓ Cached representations saved to {cache_file} (Shape: {X_all.shape})")

# 3. Paper Section IV-B Split: 70% Train (463), 10% Val (65), 20% Test (134)
indices = np.arange(len(y_all))
train_val_idx, test_idx = train_test_split(
    indices, test_size=134, random_state=42, stratify=y_all
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=65, random_state=42, stratify=y_all[train_val_idx]
)

print(f"\nSplit Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

X_tr, y_tr = X_all[train_idx], y_all[train_idx]
X_va, y_va = X_all[val_idx], y_all[val_idx]
X_te, y_te = X_all[test_idx], y_all[test_idx]

# 4. Linear Probe Architecture
class PaperLinearProbe(nn.Module):
    def __init__(self, in_features=4096):
        super().__init__()
        self.norm = nn.LayerNorm(in_features)
        self.fc = nn.Linear(in_features, 1)
        
    def forward(self, x):
        return self.fc(self.norm(x))

# 5. Stage 1: Validation LR Search
lr_candidates = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 2e-3, 5e-3]
best_lr = 1e-3
best_val_auc = -1.0

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr.float().unsqueeze(1)),
    batch_size=32,
    shuffle=True
)

for lr in lr_candidates:
    probe = PaperLinearProbe(4096).to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(100):
        probe.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(probe(bx), by)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
    probe.eval()
    with torch.no_grad():
        v_probs = torch.sigmoid(probe(X_va.to(device))).cpu().squeeze().numpy()
        
    val_auc = roc_auc_score(y_va.numpy(), v_probs)
    print(f"LR {lr:8.1e} | Val AUROC: {val_auc:.4f}")
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_lr = lr

print(f"\n✓ Best LR selected: {best_lr} (Val AUROC: {best_val_auc:.4f})")

# 6. Stage 2: Retrain on Train+Val (528 Scans) & Evaluate on Test Set (134 Scans)
X_tr_val = torch.cat([X_tr, X_va], dim=0)
y_tr_val = torch.cat([y_tr, y_va], dim=0)

full_train_loader = DataLoader(
    TensorDataset(X_tr_val, y_tr_val.float().unsqueeze(1)),
    batch_size=32,
    shuffle=True
)

final_probe = PaperLinearProbe(4096).to(device)
optimizer = torch.optim.AdamW(final_probe.parameters(), lr=best_lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(100):
    final_probe.train()
    for bx, by in full_train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(final_probe(bx), by)
        loss.backward()
        optimizer.step()
    scheduler.step()

final_probe.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(final_probe(X_te.to(device))).cpu().squeeze().numpy()

y_test_np = y_te.numpy()
final_auroc = roc_auc_score(y_test_np, test_probs)
final_auprc = average_precision_score(y_test_np, test_probs)

print("\n" + "="*62)
print("     OFFICIAL TABLE IV REPRODUCTION (SHENZHEN TEST SET)")
print("="*62)
print(f"Linear Probe AUROC : {final_auroc:.4f}  (Table IV Report: 0.967 [0.935 - 0.990])")
print(f"Linear Probe AUPRC : {final_auprc:.4f}  (Table IV Report: 0.974 [0.949 - 0.992])")
print("="*62)

Device: cuda
Cache not found in this environment. Extracting features from scratch...
✓ Backbone loaded and frozen.


Precomputing CLS Features: 100%|██████████| 42/42 [02:56<00:00,  4.19s/it]


✓ Cached representations saved to /kaggle/working/shenzhen_cls_features.pt (Shape: torch.Size([662, 4096]))

Split Sizes -> Train: 463 | Val: 65 | Test: 134
LR  1.0e-05 | Val AUROC: 0.8646
LR  5.0e-05 | Val AUROC: 0.8873
LR  1.0e-04 | Val AUROC: 0.9062
LR  5.0e-04 | Val AUROC: 0.9328
LR  1.0e-03 | Val AUROC: 0.9432
LR  2.0e-03 | Val AUROC: 0.9451
LR  5.0e-03 | Val AUROC: 0.9527

✓ Best LR selected: 0.005 (Val AUROC: 0.9527)

     OFFICIAL TABLE IV REPRODUCTION (SHENZHEN TEST SET)
Linear Probe AUROC : 0.8866  (Table IV Report: 0.967 [0.935 - 0.990])
Linear Probe AUPRC : 0.9100  (Table IV Report: 0.974 [0.949 - 0.992])
